# Notebook 5: Đánh giá trên FLEURS (Systems A, B, C)

Một notebook cho cả ba hệ thống, cùng một bảng id và cùng một cách chấm điểm:

- **A:** PhoWhisper-small → OPUS-MT vi-en gốc (pretrained).
- **B:** PhoWhisper-small → OPUS-MT fine-tune trên giả thuyết ASR (`B_100`). Các mô hình `clean_100`, `B_50`, `B_25` dùng cho ablation.
- **C:** SeamlessM4T-v2-large, dịch trực tiếp từ audio sang tiếng Anh (zero-shot).

Dữ liệu: toàn bộ FLEURS vi (train + validation + test = 4,212 bản ghi). Điểm chính tính trên bản ghi ≤ 30 giây; điểm phụ tính trên tất cả. Mọi giai đoạn lưu kết quả ra Drive và bỏ qua phần đã xong, nên chạy lại notebook sau khi phiên bị ngắt sẽ làm tiếp.

**Thứ tự:** ô 4 (zip audio) chạy được trên runtime CPU; từ ô 5 trở đi cần GPU. Nếu chạy ô 4 trên CPU, notebook chỉ tạo zip rồi dừng ở ô 5.

## 0. Môi trường

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip -q install -U transformers accelerate soundfile librosa sacrebleu rapidfuzz sentencepiece

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 which is incompatible.


## 1. Tham số

In [3]:
import os, re, json, time, gc, shutil, zipfile, unicodedata, random
from math import ceil
import numpy as np, pandas as pd, soundfile as sf, librosa, torch
from concurrent.futures import ThreadPoolExecutor

# ---------- ĐƯỜNG DẪN ----------
ROOT        = '/content/drive/MyDrive/project_DPL'
META_CSV    = os.path.join(ROOT, 'datasets/fleurs_vi_en/fleurs_vi_voice_vi_en.csv')
AUDIO_DRIVE = os.path.join(ROOT, 'datasets/fleurs_vi_en/audio')            # có 3 thư mục con: train, validation, test
ZIP_DIR     = os.path.join(ROOT, 'datasets/fleurs_vi_en/audio_zip')        # nơi lưu zip (giữ qua các phiên)
ASR_DIR     = os.path.join(ROOT, 'models/PhoWhisper-small')                # chưa có thì tải một lần từ ASR_ID
MT_BASE     = os.path.join(ROOT, 'models/opus-mt-vi-en')                   # OPUS-MT gốc (đã có từ notebook MT)
FT_DIR      = os.path.join(ROOT, 'results/ft_out')                         # các best_model đã fine-tune
OUT_DIR     = os.path.join(ROOT, 'results/fleurs_out')                     # kết quả notebook này (PHẢI nằm trên Drive)
ASR_ID, SEAMLESS_ID = 'vinai/PhoWhisper-small', 'facebook/seamless-m4t-v2-large'
AUDIO_LOCAL, ZIP_LOCAL = '/content/fleurs_audio', '/content/fleurs_zip'

# ---------- THAM SỐ ----------
SEED        = 42
SR          = 16000
MAX_SEC     = 30         # Whisper nhận tối đa 30 s. Mọi hệ thống (A/B/C) đều chỉ nghe 30 s đầu để công bằng
ASR_BEAM    = 3          # đã chốt: giống ASR của VietSpeech
MT_BEAM     = 4          # beam của OPUS-MT (A và B dùng chung)
C_BEAM      = 4          # beam của Seamless
ASR_BATCH   = 24         # như notebook ASR
MT_BATCH    = 64
C_BATCH     = 8          # Seamless 2.3 B tham số: chưa đo bộ nhớ trên T4; giảm xuống 4 hoặc 2 nếu báo hết bộ nhớ
ASR_CHUNK, C_CHUNK = 500, 250   # số bản ghi mỗi file lưu tiến trình
READ_WORKERS = 8
N_BOOT      = 1000       # số lần lấy mẫu lại cho khoảng tin cậy 95%

MT_MODELS = [            # (tên, thư mục mô hình)
    ('A',         MT_BASE),
    ('B_100',     os.path.join(FT_DIR, 'B_100',     'best_model')),
    ('clean_100', os.path.join(FT_DIR, 'clean_100', 'best_model')),
    ('B_50',      os.path.join(FT_DIR, 'B_50',      'best_model')),
    ('B_25',      os.path.join(FT_DIR, 'B_25',      'best_model')),
]

os.makedirs(OUT_DIR, exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE, '|', torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'CPU')

device: cuda | Tesla T4


## 2. Metadata và audio (zip: tạo nếu chưa có)
Đường dẫn trong CSV có tiền tố cũ (`dpl_tem/...`) nên không dùng; audio được tìm theo `split` + tên file. Zip chứa các file với tên `split/tên_file`.

In [4]:
def atomic_csv(df_, path):
    tmp = path + '.tmp'; df_.to_csv(tmp, index=False); os.replace(tmp, path)
def atomic_json(obj, path):
    tmp = path + '.tmp'; json.dump(obj, open(tmp, 'w')); os.replace(tmp, path)

df = pd.read_csv(META_CSV, keep_default_na=False)
for c in ['split', 'fleurs_id', 'audio_path', 'vi_script', 'en_script', 'duration_sec']: assert c in df.columns, f'Thiếu cột {c}'
df['fname'] = df.audio_path.map(lambda p: os.path.basename(str(p).replace('\\', '/')))
df['uid'] = df.split + '/' + df.fname
assert df.uid.is_unique, 'uid (split/tên file) không duy nhất'
assert set(df.split) <= {'train', 'validation', 'test'}, set(df.split)
assert (df.vi_script.str.strip() != '').all() and (df.en_script.str.strip() != '').all(), 'Có câu tham chiếu rỗng'
df['duration_sec'] = df.duration_sec.astype(float)
df['cluster'] = df.split + '|' + df.fleurs_id.astype(str)      # đơn vị lấy mẫu lại: câu duy nhất
df['long'] = df.duration_sec > MAX_SEC                          # bản ghi dài hơn 30 s (bị cắt)
df = df.reset_index(drop=True)
print(f'{len(df):,} bản ghi | {df.cluster.nunique():,} câu duy nhất | dài hơn {MAX_SEC}s: {int(df.long.sum())}')
print(df.groupby('split').agg(ban_ghi=('uid', 'size'), cau=('cluster', 'nunique')).to_string())

os.makedirs(ZIP_DIR, exist_ok=True); os.makedirs(ZIP_LOCAL, exist_ok=True); os.makedirs(AUDIO_LOCAL, exist_ok=True)
ZIP_NAME  = f'fleurs_vi_n{len(df)}.zip'
ZIP_DRIVE = os.path.join(ZIP_DIR, ZIP_NAME)
MEMBERS   = df.uid.tolist()

def zip_ok(path):
    try:
        with zipfile.ZipFile(path) as z: return set(z.namelist()) == set(MEMBERS) and len(z.namelist()) == len(MEMBERS)
    except Exception:
        return False

def make_zip():
    t0 = time.time(); zl = os.path.join(ZIP_LOCAL, ZIP_NAME)
    def rd(u):
        for a in range(3):
            try:
                with open(os.path.join(AUDIO_DRIVE, u), 'rb') as fh: return u, fh.read()
            except Exception:
                time.sleep(1 + a)
        return u, None
    missing = []
    with zipfile.ZipFile(zl + '.part', 'w', zipfile.ZIP_STORED) as z, ThreadPoolExecutor(READ_WORKERS) as ex:
        for i in range(0, len(MEMBERS), 200):
            for u, b in ex.map(rd, MEMBERS[i:i + 200]):
                if b is None: missing.append(u)
                else: z.writestr(zipfile.ZipInfo(u), b, compress_type=zipfile.ZIP_STORED)
            if (i // 200) % 5 == 4: print(f'   zip: {i + 200:,}/{len(MEMBERS):,} file ({time.time() - t0:.0f}s)')
    if missing:
        os.remove(zl + '.part'); raise FileNotFoundError(f'Thiếu {len(missing)} file audio trên Drive, ví dụ {missing[:3]}')
    os.replace(zl + '.part', zl); assert zip_ok(zl), 'Zip vừa tạo không hợp lệ'
    tmp = ZIP_DRIVE + '.part'; shutil.copyfile(zl, tmp); assert os.path.getsize(tmp) == os.path.getsize(zl)
    os.replace(tmp, ZIP_DRIVE)
    print(f'Đã tạo {ZIP_NAME}: {os.path.getsize(zl) / 2**30:.2f} GB, {time.time() - t0:.0f}s')

def local_ready(): return all(os.path.exists(os.path.join(AUDIO_LOCAL, u)) for u in MEMBERS)

for _s in ['train', 'validation', 'test']:
    _f = df.uid[df.split == _s].iloc[0]
    assert os.path.exists(os.path.join(AUDIO_DRIVE, _f)), f'Không thấy {os.path.join(AUDIO_DRIVE, _f)}: kiểm tra AUDIO_DRIVE'
if not (os.path.exists(ZIP_DRIVE) and zip_ok(ZIP_DRIVE)):
    print('Chưa có zip hợp lệ trên Drive: đang tạo (chạy một lần).'); make_zip()
else:
    print('Zip đã có trên Drive:', ZIP_DRIVE)

if DEVICE == 'cpu':
    print('Runtime CPU: zip đã sẵn sàng. Đổi sang GPU rồi chạy lại từ ô 0 để làm các bước tiếp theo.')
elif local_ready():
    print('Audio đã có trên ổ local, bỏ qua giải nén.')
else:
    t0 = time.time(); zl = os.path.join(ZIP_LOCAL, ZIP_NAME)
    if not (os.path.exists(zl) and zip_ok(zl)): shutil.copyfile(ZIP_DRIVE, zl)
    with zipfile.ZipFile(zl) as z: z.extractall(AUDIO_LOCAL)
    os.remove(zl); assert local_ready(), 'Giải nén không đủ file'
    print(f'Đã chép và giải nén {len(MEMBERS):,} file ({time.time() - t0:.0f}s)')

4,212 bản ghi | 1,978 câu duy nhất | dài hơn 30s: 13
            ban_ghi   cau
split                    
test            857   347
train          2994  1482
validation      361   149
Zip đã có trên Drive: /content/drive/MyDrive/project_DPL/datasets/fleurs_vi_en/audio_zip/fleurs_vi_n4212.zip
Đã chép và giải nén 4,212 file (105s)


## 3. Hàm dùng chung

In [5]:
def need_gpu():
    assert DEVICE == 'cuda', 'Cần runtime GPU cho giai đoạn này. (Ô 4 trên CPU chỉ để tạo zip.)'

def free_gpu():
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

def sync():
    if torch.cuda.is_available(): torch.cuda.synchronize()

def load_wav(uid):
    # trả về (sóng 16 kHz mono đã cắt còn MAX_SEC giây, độ dài gốc tính bằng giây, lỗi)
    try:
        w, sr = sf.read(os.path.join(AUDIO_LOCAL, uid), dtype='float32')
        if w.ndim > 1: w = w.mean(axis=1)
        if sr != SR: w = librosa.resample(w, orig_sr=sr, target_sr=SR)
        if len(w) == 0: return None, 0.0, 'empty'
        return w[:SR * MAX_SEC], len(w) / SR, ''
    except Exception as e:
        return None, 0.0, f'{type(e).__name__}: {str(e)[:60]}'

POOL = ThreadPoolExecutor(READ_WORKERS)

def run_chunked(stage, size, work):
    # Chạy work(lo, hi) -> DataFrame (không có cột uid) theo từng nhóm; nhóm đã xong thì bỏ qua.
    # Ghi thời gian từng nhóm vào timing.json (cộng dồn qua các phiên).
    d = os.path.join(OUT_DIR, stage); os.makedirs(d, exist_ok=True)
    tp = os.path.join(d, 'timing.json')
    tm = json.load(open(tp)) if os.path.exists(tp) else {'chunks': {}, 'peak_vram_gb': 0.0, 'params_m': None}
    n = len(df); parts = []
    for k in range(ceil(n / size)):
        lo, hi = k * size, min(n, (k + 1) * size); p = os.path.join(d, f'part_{k:04d}.csv')
        ids = df.uid.iloc[lo:hi].tolist(); r = None
        if os.path.exists(p):
            try:
                r = pd.read_csv(p, keep_default_na=False)
                if r.uid.tolist() != ids: r = None
            except Exception:
                r = None
        if r is None:
            if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()
            sync(); t0 = time.time()
            r = work(lo, hi); sync(); dt = time.time() - t0
            r.insert(0, 'uid', ids); atomic_csv(r, p)
            tm['chunks'][str(k)] = dt
            if torch.cuda.is_available(): tm['peak_vram_gb'] = max(tm['peak_vram_gb'], torch.cuda.max_memory_allocated() / 2**30)
            atomic_json(tm, tp)
            print(f'[{stage}] nhóm {k + 1}/{ceil(n / size)}: {dt:.0f}s')
        parts.append(r)
    out = pd.concat(parts, ignore_index=True); assert out.uid.tolist() == df.uid.tolist()
    return out, tm

def tm_total(tm): return float(sum(tm['chunks'].values()))

class AudioDS(torch.utils.data.Dataset):
    def __init__(self, wavs): self.w = wavs
    def __len__(self): return len(self.w)
    def __getitem__(self, i): return {'raw': self.w[i], 'sampling_rate': SR}

## 4. Giai đoạn ASR: PhoWhisper-small (dùng chung cho A và B)

In [6]:
need_gpu()
from transformers import pipeline
_ASR = {}

def get_asr():
    if 'pipe' in _ASR: return _ASR['pipe']
    if not os.path.isfile(os.path.join(ASR_DIR, 'config.json')):
        from huggingface_hub import snapshot_download
        tmpd = ASR_DIR + '.tmp'; shutil.rmtree(tmpd, ignore_errors=True); os.makedirs(os.path.dirname(ASR_DIR), exist_ok=True)
        snapshot_download(ASR_ID, local_dir=tmpd, ignore_patterns=['*.msgpack', '*.h5', '*.ot'])
        assert os.path.isfile(os.path.join(tmpd, 'config.json')); shutil.rmtree(ASR_DIR, ignore_errors=True); os.rename(tmpd, ASR_DIR)
    try: pipe = pipeline('automatic-speech-recognition', model=ASR_DIR, device=0, dtype=torch.float16)
    except TypeError: pipe = pipeline('automatic-speech-recognition', model=ASR_DIR, device=0, torch_dtype=torch.float16)
    _ASR['pipe'] = pipe; _ASR['params_m'] = sum(p.numel() for p in pipe.model.parameters()) / 1e6
    _ASR['gk'] = {'language': 'vi', 'task': 'transcribe', 'num_beams': ASR_BEAM}
    list(pipe(AudioDS([np.zeros(SR, dtype='float32')] * 2), batch_size=2, generate_kwargs=_ASR['gk']))   # khởi động, không tính giờ
    return pipe

def asr_work(lo, hi):
    pipe = get_asr(); rows = df.iloc[lo:hi]
    reads = list(POOL.map(load_wav, rows.uid))
    hyp, dur, err = [''] * len(rows), [r[1] for r in reads], [r[2] for r in reads]
    ok = [i for i, r in enumerate(reads) if r[0] is not None]
    if ok:
        with torch.no_grad():
            out = pipe(AudioDS([reads[i][0] for i in ok]), batch_size=ASR_BATCH, generate_kwargs=_ASR['gk'])
            for i, o in zip(ok, out): hyp[i] = o['text'].strip()
    return pd.DataFrame({'asr_text': hyp, 'dur_s': np.round(dur, 3), 'error': err})

asr, tm_asr = run_chunked('asr', ASR_CHUNK, asr_work)
if 'pipe' in _ASR and tm_asr['params_m'] is None:
    tm_asr['params_m'] = _ASR['params_m']; atomic_json(tm_asr, os.path.join(OUT_DIR, 'asr', 'timing.json'))
_ASR.clear(); free_gpu()

asr['asr_text'] = asr.asr_text.astype(str)
atomic_csv(asr, os.path.join(OUT_DIR, 'fleurs_asr.csv'))
df = df.drop(columns=[c for c in ['asr_text', 'dur_s', 'error'] if c in df.columns]).merge(asr, on='uid', how='left'); assert df.asr_text.notna().all()
bad = df[df.error != '']
print(f'ASR xong: lỗi đọc audio {len(bad)} | ASR rỗng {int((df.asr_text.str.strip() == "").sum())} | '
      f'lệch độ dài so với metadata > 0.5s: {int(((df.dur_s - df.duration_sec).abs() > 0.5).sum())}')
print(f'Thời gian ASR: {tm_total(tm_asr) / 60:.1f} phút | RTF = {tm_total(tm_asr) / np.minimum(df.dur_s, MAX_SEC).sum():.4f}')
for _, r in df.head(3).iterrows(): print(' -', r.vi_script, '|', r.asr_text)

Loading weights:   0%|          | 0/480 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'begin_suppress_tokens', 'suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


[asr] nhóm 1/9: 282s
[asr] nhóm 2/9: 253s
[asr] nhóm 3/9: 242s
[asr] nhóm 4/9: 248s
[asr] nhóm 5/9: 272s
[asr] nhóm 6/9: 261s
[asr] nhóm 7/9: 259s
[asr] nhóm 8/9: 270s
[asr] nhóm 9/9: 113s
ASR xong: lỗi đọc audio 0 | ASR rỗng 0 | lệch độ dài so với metadata > 0.5s: 0
Thời gian ASR: 36.7 phút | RTF = 0.0461
 - Theo sau quá trình tố tụng này, HJR-3 sẽ được xem xét lại bởi cơ quan lập pháp được bầu cử tiếp theo trong năm 2015 hoặc 2016 để tiếp tục quá trình xử lý. | theo sau quá trình tố tụng này hắt chi r ba sẽ được xem xét lại bởi cơ quan lập pháp được bầu cử tiếp theo trong năm hai ngàn không trăm mười lăm hoặc hai ngàn không trăm mười sáu để tiếp tục quá trình xử lý.
 - Phát minh bánh xe nan hoa làm cho những cỗ xe ngựa chiến của người Assyria nhẹ hơn, nhanh hơn và sẵn sàng hơn để đánh bại các binh lính và cỗ xe khác. | phát minh bánh xe nan hoa làm cho những cỗ xe ngựa chiến của người assyria nhẹ hơn nhanh hơn và sẵn sàng hơn để đánh bại các binh lính và cỗ xe khác.
 - Karno là một t

## 5. Giai đoạn dịch (A và B, cùng ablation)
Mỗi mô hình dịch hai đầu vào: giả thuyết ASR (`asr`, hệ thống thật) và transcript chuẩn (`gold`, dùng để đo mức lan truyền lỗi ASR).

In [8]:
need_gpu()
import warnings
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, logging

# Ẩn các cảnh báo không cần thiết từ transformers và python
logging.set_verbosity_error()
warnings.filterwarnings("ignore", category=UserWarning)

MT_MAXLEN = 128
tok = AutoTokenizer.from_pretrained(MT_BASE)
mt_out_dir = os.path.join(OUT_DIR, 'mt'); os.makedirs(mt_out_dir, exist_ok=True)
mt_tp = os.path.join(mt_out_dir, 'timing.json')
mt_tm = json.load(open(mt_tp)) if os.path.exists(mt_tp) else {}

def translate(model, texts):
    order = sorted(range(len(texts)), key=lambda i: -len(texts[i])); res = [''] * len(texts)
    for s in range(0, len(order), MT_BATCH):
        idx = order[s:s + MT_BATCH]
        enc = tok([texts[i] for i in idx], return_tensors='pt', padding=True, truncation=True, max_length=MT_MAXLEN).to('cuda')
        with torch.no_grad():
            # Thiết lập rõ ràng hoặc bỏ max_length khi đã dùng max_new_tokens để tránh xung đột cảnh báo
            out = model.generate(**enc, num_beams=MT_BEAM, max_new_tokens=160)
        for i, t in zip(idx, tok.batch_decode(out, skip_special_tokens=True)): res[i] = t.strip()
    return res

INPUTS = [('asr', 'asr_text'), ('gold', 'vi_script')]
for name, path in MT_MODELS:
    todo = []
    for inp, col in INPUTS:
        f = os.path.join(mt_out_dir, f'{name}__{inp}.csv')
        ok = False
        if os.path.exists(f):
            try: ok = pd.read_csv(f, keep_default_na=False).uid.tolist() == df.uid.tolist()
            except Exception: ok = False
        if not ok: todo.append((inp, col, f))
    if not todo: print(f'[{name}] đã xong, bỏ qua.'); continue
    assert os.path.isfile(os.path.join(path, 'config.json')), f'Không thấy mô hình: {path}'
    model = AutoModelForSeq2SeqLM.from_pretrained(path).half().to('cuda').eval()
    # cấu hình generation_config để tránh in warning trùng lặp độ dài
    if hasattr(model, 'generation_config'):
        model.generation_config.max_length = None
    params_m = sum(p.numel() for p in model.parameters()) / 1e6
    translate(model, ['xin chào'] * 2)                                   # khởi động, không tính giờ
    for inp, col, f in todo:
        torch.cuda.reset_peak_memory_stats(); sync(); t0 = time.time()
        hyp = translate(model, df[col].astype(str).tolist()); sync(); dt = time.time() - t0
        atomic_csv(pd.DataFrame({'uid': df.uid, 'hyp': hyp}), f)
        mt_tm[f'{name}__{inp}'] = {'s': dt, 'peak_vram_gb': torch.cuda.max_memory_allocated() / 2**30, 'params_m': params_m}
        atomic_json(mt_tm, mt_tp)
        print(f'[{name} | {inp}] {dt:.0f}s | VRAM đỉnh {mt_tm[f"{name}__{inp}"]["peak_vram_gb"]:.2f} GB')
    del model; free_gpu()

MT = {f'{name}__{inp}': pd.read_csv(os.path.join(mt_out_dir, f'{name}__{inp}.csv'), keep_default_na=False).hyp.astype(str).values
      for name, _ in MT_MODELS for inp, _c in INPUTS}
print('\nVí dụ (câu đầu tiên của test):')
r0 = df[df.split == 'test'].index[0]
print(' REF :', df.en_script[r0]); print(' ASR :', df.asr_text[r0])
for name, _ in MT_MODELS: print(f' {name:9s}:', MT[f'{name}__asr'][r0])

[A] đã xong, bỏ qua.
[B_100] đã xong, bỏ qua.


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

[clean_100 | asr] 56s | VRAM đỉnh 0.98 GB
[clean_100 | gold] 56s | VRAM đỉnh 1.08 GB


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

[B_50 | asr] 60s | VRAM đỉnh 1.01 GB
[B_50 | gold] 53s | VRAM đỉnh 1.02 GB


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

[B_25 | asr] 66s | VRAM đỉnh 1.15 GB
[B_25 | gold] 51s | VRAM đỉnh 0.91 GB

Ví dụ (câu đầu tiên của test):
 REF : Ancient cultures and tribes began to keep them for easy access to milk, hair, meat, and skins.
 ASR : văn hoá và bộ lạc cổ xưa đã bắt đầu giữ những con vật này để dễ lấy sữa tóc thịt và da.
 A        : Ancient culture and tribe began to hold these animals so that they could easily milk their flesh and skin.
 B_100    : Ancient culture and tribe have begun to hold these animals to ease the milk of flesh and skin.
 clean_100: Ancient culture and tribe began to hold these animals to easy access to meat and skin milk.
 B_50     : Culture and the ancient tribes began to hold these animals so that it would be easy to milk meat and leather.
 B_25     : Ancient culture and tribe began to hold these animals to ease the milk of flesh and skin.


## 6. Giai đoạn System C: SeamlessM4T-v2-large (audio → văn bản tiếng Anh)
Chưa đo bộ nhớ và tốc độ của mô hình này trên T4. Nếu hết bộ nhớ, giảm `C_BATCH` ở ô 3 rồi chạy lại ô này (các nhóm đã xong được giữ).

In [9]:
need_gpu()
from transformers import AutoProcessor, SeamlessM4Tv2ForSpeechToText
_C = {}

def get_c():
    if 'model' in _C: return _C
    _C['proc'] = AutoProcessor.from_pretrained(SEAMLESS_ID)
    try: m = SeamlessM4Tv2ForSpeechToText.from_pretrained(SEAMLESS_ID, dtype=torch.float16)
    except TypeError: m = SeamlessM4Tv2ForSpeechToText.from_pretrained(SEAMLESS_ID, torch_dtype=torch.float16)
    _C['model'] = m.to('cuda').eval(); _C['params_m'] = sum(p.numel() for p in m.parameters()) / 1e6
    return _C

def seamless_batch(wavs):
    c = get_c(); proc, model = c['proc'], c['model']
    try: feats = proc(audio=wavs, sampling_rate=SR, return_tensors='pt', padding=True)
    except TypeError: feats = proc(audios=wavs, sampling_rate=SR, return_tensors='pt', padding=True)
    feats = feats.to('cuda'); feats['input_features'] = feats['input_features'].half()
    with torch.no_grad():
        out = model.generate(**feats, tgt_lang='eng', num_beams=C_BEAM, max_new_tokens=160)
    if hasattr(out, 'sequences'): out = out.sequences
    return [t.strip() for t in proc.batch_decode(out, skip_special_tokens=True)]

def c_work(lo, hi):
    rows = df.iloc[lo:hi]; reads = list(POOL.map(load_wav, rows.uid)); hyp = [''] * len(rows)
    ok = [i for i, r in enumerate(reads) if r[0] is not None]
    for s in range(0, len(ok), C_BATCH):
        b = ok[s:s + C_BATCH]
        try:
            res = seamless_batch([reads[i][0] for i in b])
        except torch.cuda.OutOfMemoryError:
            free_gpu(); res = [seamless_batch([reads[i][0]])[0] for i in b]     # thử lại từng mẫu; nếu vẫn hết bộ nhớ thì báo lỗi
        for i, t in zip(b, res): hyp[i] = t
    return pd.DataFrame({'hyp': hyp})

cres, tm_c = run_chunked('seamless', C_CHUNK, c_work)
if 'model' in _C and tm_c['params_m'] is None:
    tm_c['params_m'] = _C['params_m']; atomic_json(tm_c, os.path.join(OUT_DIR, 'seamless', 'timing.json'))
_C.clear(); free_gpu()
MT['C'] = cres.hyp.astype(str).values
print(f'Seamless xong: {tm_total(tm_c) / 60:.1f} phút | RTF = {tm_total(tm_c) / np.minimum(df.dur_s, MAX_SEC).sum():.4f} | rỗng: {int((cres.hyp.str.strip() == "").sum())}')
r0 = df[df.split == 'test'].index[0]; print(' C   :', MT['C'][r0])

preprocessor_config.json:   0%|          | 0.00/1.78k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/2.72k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/19.7k [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B / 5.17MB            

tokenizer.model: downloading bytes:           |  0.00B            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1429 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/9.91M [00:00<?, ?B/s]

[seamless] nhóm 1/17: 293s
[seamless] nhóm 2/17: 116s
[seamless] nhóm 3/17: 121s
[seamless] nhóm 4/17: 115s
[seamless] nhóm 5/17: 110s
[seamless] nhóm 6/17: 118s
[seamless] nhóm 7/17: 114s
[seamless] nhóm 8/17: 115s
[seamless] nhóm 9/17: 117s
[seamless] nhóm 10/17: 119s
[seamless] nhóm 11/17: 122s
[seamless] nhóm 12/17: 109s
[seamless] nhóm 13/17: 129s
[seamless] nhóm 14/17: 132s
[seamless] nhóm 15/17: 135s
[seamless] nhóm 16/17: 140s
[seamless] nhóm 17/17: 114s
Seamless xong: 37.0 phút | RTF = 0.0465 | rỗng: 0
 C   : Ancient cultures and tribes began keeping these animals for milk, hair, meat, and skin.


## 7. Chấm điểm
- **WER/CER** của PhoWhisper: chuẩn hóa (NFC, chữ thường, bỏ dấu câu) rồi so với `vi_script`.
- **SacreBLEU** (mặc định, phân biệt hoa thường) và **chrF++** so với `en_script`.
- Khoảng tin cậy 95% bằng bootstrap **theo câu duy nhất** (các bản ghi cùng một câu được lấy mẫu cùng nhau); độ chênh giữa hai hệ thống dùng cùng các lần lấy mẫu (bootstrap cặp).
- Hai nhóm bản ghi: `main` = ≤ 30 giây (điểm chính), `all` = tất cả (13 bản ghi dài bị cắt còn 30 s).

In [10]:
from sacrebleu.metrics import BLEU, CHRF
from rapidfuzz.distance import Levenshtein as Lev
BLEU_M, CHRF_M = BLEU(), CHRF(word_order=2)           # chrF++ = chrF với word_order=2

def norm(s):
    s = unicodedata.normalize('NFC', str(s)).lower()
    s = re.sub(r'[^\w\s]|_', ' ', s)
    return re.sub(r'\s+', ' ', s).strip()

# ---- thống kê theo từng câu ----
ref_w = [norm(s).split() for s in df.vi_script]; ref_c = [list(norm(s)) for s in df.vi_script]
hyp_w = [norm(s).split() for s in df.asr_text];  hyp_c = [list(norm(s)) for s in df.asr_text]
ASR_STATS = np.array([[Lev.distance(a, b), len(a), Lev.distance(c, d), len(c)]
                      for a, b, c, d in zip(ref_w, hyp_w, ref_c, hyp_c)], dtype=float)    # lỗi từ, số từ, lỗi ký tự, số ký tự

refs = df.en_script.astype(str).tolist()
def mt_stats(hyps):
    hyps = [str(h) for h in hyps]
    return (np.array(BLEU_M._extract_corpus_statistics(hyps, [refs]), dtype=float),
            np.array(CHRF_M._extract_corpus_statistics(hyps, [refs]), dtype=float))

SYS = {'A': MT['A__asr'], 'B': MT['B_100__asr'], 'C': MT['C'],
       'clean_100': MT['clean_100__asr'], 'B_50': MT['B_50__asr'], 'B_25': MT['B_25__asr']}
for n_, _p in MT_MODELS: SYS[f'{n_} [gold]'] = MT[f'{n_}__gold']        # đầu vào là transcript chuẩn (không có lỗi ASR)
REFSYS = {'A': None, 'B': 'A', 'C': 'A', 'clean_100': 'B', 'B_50': 'B', 'B_25': 'B',
          'A [gold]': 'A', 'B_100 [gold]': 'B', 'clean_100 [gold]': 'clean_100', 'B_50 [gold]': 'B_50', 'B_25 [gold]': 'B_25'}
STATS = {k: mt_stats(v) for k, v in SYS.items()}

def sc_bleu(s): return BLEU_M._compute_score_from_stats(list(s)).score
def sc_chrf(s): return CHRF_M._compute_score_from_stats(list(s)).score

def cluster_sum(M, cl):
    return pd.DataFrame(M).groupby(cl.values).sum().values        # cộng thống kê theo câu duy nhất

def ci(x): return np.percentile(x, [2.5, 97.5])

VIEWS = {'pooled_main': (df.duration_sec <= MAX_SEC), 'pooled_all': pd.Series(True, index=df.index),
         'test_main': (df.split == 'test') & (df.duration_sec <= MAX_SEC),
         'validation_main': (df.split == 'validation') & (df.duration_sec <= MAX_SEC),
         'train_main': (df.split == 'train') & (df.duration_sec <= MAX_SEC)}

def eval_mt(view):
    m = VIEWS[view].values; cl = df.cluster[m]
    G = {k: (cluster_sum(STATS[k][0][m], cl), cluster_sum(STATS[k][1][m], cl)) for k in SYS}
    ncl = len(next(iter(G.values()))[0]); rng = np.random.default_rng(SEED)
    idx = rng.integers(0, ncl, size=(N_BOOT, ncl))                 # cùng lần lấy mẫu cho mọi hệ thống
    point, boot = {}, {}
    for k, (b, c) in G.items():
        point[k] = (sc_bleu(b.sum(0)), sc_chrf(c.sum(0)))
        boot[k] = (np.array([sc_bleu(b[i].sum(0)) for i in idx]), np.array([sc_chrf(c[i].sum(0)) for i in idx]))
    rows = []
    for k in SYS:
        r = {'view': view, 'system': k, 'n_rec': int(m.sum()), 'n_sent': ncl,
             'BLEU': point[k][0], 'BLEU_lo': ci(boot[k][0])[0], 'BLEU_hi': ci(boot[k][0])[1],
             'chrF++': point[k][1], 'chrF_lo': ci(boot[k][1])[0], 'chrF_hi': ci(boot[k][1])[1], 'ref': REFSYS.get(k)}
        ref = REFSYS.get(k)
        if ref:
            for j, nm in enumerate(['dBLEU', 'dchrF++']):
                d = boot[k][j] - boot[ref][j]
                r[nm] = point[k][j] - point[ref][j]; r[nm + '_lo'], r[nm + '_hi'] = ci(d)
        rows.append(r)
    return pd.DataFrame(rows)

def eval_asr(view):
    m = VIEWS[view].values; S = cluster_sum(ASR_STATS[m], df.cluster[m]); rng = np.random.default_rng(SEED)
    idx = rng.integers(0, len(S), size=(N_BOOT, len(S)))
    bs = np.array([S[i].sum(0) for i in idx]); tot = S.sum(0)
    w, c = 100 * tot[0] / tot[1], 100 * tot[2] / tot[3]; wb, cb = 100 * bs[:, 0] / bs[:, 1], 100 * bs[:, 2] / bs[:, 3]
    return {'view': view, 'n_rec': int(m.sum()), 'n_sent': len(S), 'WER': w, 'WER_lo': ci(wb)[0], 'WER_hi': ci(wb)[1],
            'CER': c, 'CER_lo': ci(cb)[0], 'CER_hi': ci(cb)[1]}

TAB = os.path.join(OUT_DIR, 'tables'); os.makedirs(TAB, exist_ok=True)
t_asr = pd.DataFrame([eval_asr(v) for v in VIEWS]); atomic_csv(t_asr, os.path.join(TAB, 'asr_wer_cer.csv'))
t_mt  = pd.concat([eval_mt(v) for v in VIEWS], ignore_index=True); atomic_csv(t_mt, os.path.join(TAB, 'mt_scores.csv'))

pd.set_option('display.width', 220); pd.set_option('display.max_columns', 30); pd.options.display.float_format = '{:.2f}'.format
print('== ASR (PhoWhisper-small, beam', ASR_BEAM, ') ==');  print(t_asr.to_string(index=False))
cols = ['system', 'BLEU', 'BLEU_lo', 'BLEU_hi', 'chrF++', 'chrF_lo', 'chrF_hi', 'dBLEU', 'dBLEU_lo', 'dBLEU_hi', 'dchrF++', 'ref']
for v in ['pooled_main', 'pooled_all', 'test_main']:
    print(f'\n== Dịch: {v} (Δ so với cột ref; [gold] so với cùng mô hình có lỗi ASR) =='); print(t_mt[t_mt['view'] == v][cols].to_string(index=False))

== ASR (PhoWhisper-small, beam 3 ) ==
           view  n_rec  n_sent   WER  WER_lo  WER_hi  CER  CER_lo  CER_hi
    pooled_main   4199    1974 11.36   10.74   11.99 7.62    7.14    8.10
     pooled_all   4212    1978 11.54   10.89   12.20 7.78    7.28    8.29
      test_main    850     346 12.16   10.84   13.51 7.61    6.61    8.61
validation_main    359     148 10.76    8.80   13.06 6.83    5.34    8.63
     train_main   2990    1480 11.20   10.40   11.98 7.72    7.11    8.35

== Dịch: pooled_main (Δ so với cột ref; [gold] so với cùng mô hình có lỗi ASR) ==
          system  BLEU  BLEU_lo  BLEU_hi  chrF++  chrF_lo  chrF_hi  dBLEU  dBLEU_lo  dBLEU_hi  dchrF++       ref
               A 16.33    15.74    16.96   42.63    42.13    43.14    NaN       NaN       NaN      NaN      None
               B 17.24    16.64    17.88   44.34    43.85    44.86   0.91      0.50      1.35     1.71         A
               C 27.75    27.04    28.48   52.69    52.15    53.24  11.42     10.74     12.14   

## 8. Hiệu năng

In [11]:
audio_s = float(np.minimum(df.dur_s, MAX_SEC).sum())
rows = []
asr_s = tm_total(tm_asr)
rows.append({'thành phần': 'PhoWhisper-small (ASR)', 'tham số (M)': tm_asr['params_m'], 'thời gian (s)': asr_s, 'VRAM đỉnh (GB)': tm_asr['peak_vram_gb']})
for name, _ in MT_MODELS:
    t = mt_tm[f'{name}__asr']; rows.append({'thành phần': f'OPUS-MT {name} (dịch, đầu vào ASR)', 'tham số (M)': t['params_m'], 'thời gian (s)': t['s'], 'VRAM đỉnh (GB)': t['peak_vram_gb']})
rows.append({'thành phần': 'Seamless-v2-large (C, trọn gói)', 'tham số (M)': tm_c['params_m'], 'thời gian (s)': tm_total(tm_c), 'VRAM đỉnh (GB)': tm_c['peak_vram_gb']})
eff = pd.DataFrame(rows)
eff['ms/bản ghi'] = 1000 * eff['thời gian (s)'] / len(df); eff['RTF'] = eff['thời gian (s)'] / audio_s
a_pipe, c_pipe = asr_s + mt_tm['A__asr']['s'], tm_total(tm_c)
pipe_tab = pd.DataFrame([{'hệ thống': 'A (ASR + OPUS-MT gốc)', 'thời gian (s)': a_pipe, 'RTF': a_pipe / audio_s, 'ms/bản ghi': 1000 * a_pipe / len(df)},
                         {'hệ thống': 'B (ASR + OPUS-MT fine-tune)', 'thời gian (s)': asr_s + mt_tm['B_100__asr']['s'], 'RTF': (asr_s + mt_tm['B_100__asr']['s']) / audio_s, 'ms/bản ghi': 1000 * (asr_s + mt_tm['B_100__asr']['s']) / len(df)},
                         {'hệ thống': 'C (Seamless)', 'thời gian (s)': c_pipe, 'RTF': c_pipe / audio_s, 'ms/bản ghi': 1000 * c_pipe / len(df)}])
atomic_csv(eff, os.path.join(TAB, 'efficiency_components.csv')); atomic_csv(pipe_tab, os.path.join(TAB, 'efficiency_pipelines.csv'))
print(f'Tổng audio: {audio_s / 3600:.2f} giờ ({len(df):,} bản ghi, đã cắt {MAX_SEC}s); chạy theo batch trên một GPU, nên đây là thông lượng, không phải độ trễ từng câu.')
print(eff.to_string(index=False)); print(); print(pipe_tab.to_string(index=False))
print('\nThời gian fine-tune của B (từ notebook MT): xem results/ft_out/*/timing.json')

Tổng audio: 13.26 giờ (4,212 bản ghi, đã cắt 30s); chạy theo batch trên một GPU, nên đây là thông lượng, không phải độ trễ từng câu.
                           thành phần  tham số (M)  thời gian (s)  VRAM đỉnh (GB)  ms/bản ghi  RTF
               PhoWhisper-small (ASR)       241.73        2200.45            5.02      522.42 0.05
        OPUS-MT A (dịch, đầu vào ASR)        72.18          60.06            0.96       14.26 0.00
    OPUS-MT B_100 (dịch, đầu vào ASR)        72.18          61.92            1.01       14.70 0.00
OPUS-MT clean_100 (dịch, đầu vào ASR)        72.18          55.97            0.98       13.29 0.00
     OPUS-MT B_50 (dịch, đầu vào ASR)        72.18          60.13            1.01       14.28 0.00
     OPUS-MT B_25 (dịch, đầu vào ASR)        72.18          65.86            1.15       15.64 0.00
      Seamless-v2-large (C, trọn gói)      1501.84        2220.33            5.45      527.14 0.05

                   hệ thống  thời gian (s)  RTF  ms/bản ghi
      A (ASR +

## 9. Gộp đầu ra từng bản ghi để phân tích lỗi

In [12]:
full = df[['uid', 'split', 'fleurs_id', 'duration_sec', 'long', 'vi_script', 'en_script', 'asr_text']].copy()
for k, v in SYS.items(): full[f'hyp_{k}'] = v
atomic_csv(full, os.path.join(OUT_DIR, 'fleurs_all_outputs.csv'))
print(f'Đã ghi {len(full):,} dòng vào fleurs_all_outputs.csv (mỗi hệ thống một cột hyp_*).')
try:
    from google.colab import drive
    drive.flush_and_unmount(); print('Đã flush Drive.')
except Exception as e:
    print('Bỏ qua flush:', e)

Đã ghi 4,212 dòng vào fleurs_all_outputs.csv (mỗi hệ thống một cột hyp_*).
Đã flush Drive.
